In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 텐서 네트워크 오류 완화(TEM): Algorithmiq의 Qiskit Function
> **Note:** Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API 경유) Plan 사용자에게만 제공되는 실험적 기능입니다. 현재 미리 보기 릴리스 상태이며 변경될 수 있습니다.


<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하시기 바랍니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## 개요
Algorithmiq의 텐서 네트워크 오류 완화(TEM) 방법은 노이즈 완화를
클래식 후처리 단계에서 전적으로 수행하도록 설계된 하이브리드
양자-클래식 알고리즘입니다. TEM을 사용하면 정확도와 비용 효율성을 높여
양자 하드웨어에서 불가피하게 발생하는 노이즈 유발 오류를 완화하면서
관측값의 기댓값을 계산할 수 있으며, 이는 양자 연구자와 산업
실무자 모두에게 매우 매력적인 옵션입니다.

이 방법은 양자 프로세서의 상태에 영향을 미치는 전역 노이즈 채널의
역행렬을 나타내는 텐서 네트워크를 구성한 다음, 노이즈가 있는 상태에서
획득한 정보 완전 측정 결과에 해당 맵을 적용하여 관측값에 대한
비편향 추정값을 얻는 방식으로 이루어집니다.

TEM의 장점으로는 정보 완전 측정을 활용하여 광범위한 관측값의 완화된
기댓값에 접근할 수 있으며, Filippov et al. (2023),
[arXiv:2307.11740](https://arxiv.org/abs/2307.11740) 및 Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542)에
설명된 것처럼 양자 하드웨어에서 최적의 샘플링 오버헤드를 가집니다.
측정 오버헤드는 효율적인 오류 완화를 수행하는 데 필요한 추가 측정
횟수를 의미하며, 양자 계산의 실현 가능성에 있어 핵심적인 요소입니다.
따라서 TEM은 양자 카오스, 다체 물리학, 허바드 동역학, 소분자 화학
시뮬레이션 등 복잡한 분야의 응용에서 양자 우위를 가능하게 할
잠재력을 지니고 있습니다.

TEM의 주요 특징과 이점을 요약하면 다음과 같습니다:

1.  **최적의 측정 오버헤드**: TEM은
[이론적 한계](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601)에
대해 최적화되어 있으며, 어떠한 방법도 더 작은 측정 오버헤드를 달성할 수
없음을 의미합니다. 즉, TEM은 오류 완화를 수행하는 데 필요한 최소한의
추가 측정만을 요구합니다. 이는 곧 TEM이 최소한의 양자 런타임을 사용함을
의미합니다.
2.  **비용 효율성**: TEM은 노이즈 완화를 후처리 단계에서 전적으로 처리하므로
양자 컴퓨터에 추가 Circuit을 삽입할 필요가 없어 계산 비용을 절감할 뿐만
아니라 양자 디바이스의 불완전성으로 인한 추가 오류 발생 위험도 줄입니다.
3.  **다중 관측값 추정**: 정보 완전 측정 덕분에 TEM은 양자 컴퓨터로부터 동일한
측정 데이터로 여러 관측값을 효율적으로 추정합니다.
4.  **측정 오류 완화**: TEM Qiskit Function에는 짧은 보정 실행 후 판독 오류를
크게 줄일 수 있는 [독자적인 측정 오류 완화 방법](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)도
포함되어 있습니다.
5.  **정확도**: TEM은 디지털 양자 시뮬레이션의 정확도와 신뢰성을 크게 향상시켜
양자 알고리즘을 더욱 정밀하고 신뢰할 수 있게 만듭니다.
## 설명
TEM 함수를 사용하면 최소한의 샘플링 오버헤드로 양자 Circuit에서 여러
관측값에 대한 오류 완화된 기댓값을 얻을 수 있습니다. Circuit은 정보 완전
양의 연산자 값 측정(IC-POVM)으로 측정되며, 수집된 측정 결과는 클래식
컴퓨터에서 처리됩니다. 이 측정은 텐서 네트워크 방법을 수행하고
노이즈 역전 맵을 구축하는 데 사용됩니다. 함수는 텐서 네트워크를 사용하여
노이즈 레이어를 표현하고 전체 노이즈 Circuit을 완전히 역전하는 맵을
적용합니다.

![TEM 개략도](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "노이즈가 있는 양자 프로세서의 측정 결과를 후처리하여 관측값 O의 오류 완화된 추정. U와 N은 이상적인 양자 연산과 관련된 노이즈 맵을 나타내며, 일반적으로 비국소적일 수 있습니다(회색 박스로 확장). D는 IC 측정의 효과에 이중인 연산자 텐서를 나타냅니다. 노이즈 완화 모듈 M은 중간에서 밖으로 효율적으로 수축되는 텐서 네트워크입니다. 수축의 첫 번째 반복은 점선 보라색 선으로, 두 번째는 파선으로, 세 번째는 실선으로 표시됩니다.")

Circuit이 함수에 제출되면 두 Qubit 게이트(양자 디바이스에서 더 노이즈가
많은 Gate)의 레이어 수를 최소화하도록 변환 및 최적화됩니다. 레이어에
영향을 미치는 노이즈는 E. van den Berg, Z. Minev, A. Kandala, K. Temme,
Nat. Phys. (2023), [arXiv:2201.09866](https://arxiv.org/abs/2201.09866)에
설명된 희소 Pauli-Lindblad 노이즈 모델을 사용하여
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)을
통해 학습됩니다.

노이즈 모델은 Qubit 크로스토크를 포함한 미묘한 특징을 포착할 수 있는
디바이스 노이즈에 대한 정확한 설명입니다. 그러나 디바이스의 노이즈는
변동하고 드리프트할 수 있으며, 학습된 노이즈는 추정이 수행되는 시점에서
정확하지 않을 수 있습니다. 이로 인해 부정확한 결과가 발생할 수 있습니다.
## 시작하기
[IBM Quantum Platform API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하고, 다음과 같이 TEM 함수를 선택하세요. (이 스니펫은 이미 [계정을 로컬 환경에 저장](/guides/functions#install-qiskit-functions-catalog-client)했다고 가정합니다.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## 예제
다음 스니펫은 TEM을 사용하여 간단한 양자 Circuit에 주어진 관측값의 기댓값을 계산하는 예제를 보여줍니다.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Qiskit Serverless API를 사용하여 Qiskit Function 워크로드의 상태를 확인하세요:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

다음과 같이 결과를 반환할 수 있습니다:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** 주어진 연산자에 대한 노이즈 없는 Circuit의 기댓값은 `0.18409094298943401` 근처여야 합니다.
## 입력
**매개변수**

이름 | 타입 | 설명 | 필수 여부 | 기본값 | 예제
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | `(circuit, observables)` 또는 `(circuit, observables, parameters, precision)`과 같은 튜플 형태의 PUB 유사(기본 통합 블록) 객체의 이터러블입니다. 자세한 내용은 [PUB 개요](/guides/primitive-input-output#overview-of-pubs)를 참조하세요. 비-ISA Circuit이 전달되면 최적 설정으로 변환됩니다. ISA Circuit이 전달되면 변환되지 않으며, 이 경우 관측값은 전체 QPU에 대해 정의되어야 합니다. | 예 | 없음 | (circuit, observables)
`backend_name` | str | 쿼리를 수행할 Backend의 이름입니다. | 아니요 | 제공하지 않으면 가장 여유 있는 Backend가 사용됩니다. | "ibm_fez"
`options` | dict | 입력 옵션입니다. 자세한 내용은 `Options` 섹션을 참조하세요. | 아니요 | 자세한 내용은 `Options` 섹션을 참조하세요. | {"max_bond_dimension": 100\}

> **Caution:** TEM에는 현재 다음과 같은 제한 사항이 있습니다:
> 
>   - 매개변수화된 Circuit은 지원되지 않습니다. precision이 지정된 경우 parameters 인수는 `None`으로 설정해야 합니다. 이 제한은 향후 버전에서 제거될 예정입니다.
>   - 루프가 없는 Circuit만 지원됩니다. 이 제한은 향후 버전에서 제거될 예정입니다.
>   - reset, measure, 모든 형태의 제어 흐름과 같은 비유니터리 Gate는 지원되지 않습니다. reset에 대한 지원은 향후 릴리스에서 추가될 예정입니다.
### 옵션
TEM의 고급 옵션을 포함하는 딕셔너리입니다. 딕셔너리에는 아래 표의 키가 포함될 수 있습니다. 옵션이 제공되지 않으면 표에 나열된 기본값이 사용됩니다. 기본값은 TEM의 일반적인 사용에 적합합니다.

이름 | 선택값 | 설명 | 기본값
-- | -- | -- | --
`tem_max_bond_dimension` | int | 텐서 네트워크에 사용할 최대 본드 차원입니다. | 500 |
`tem_compression_cutoff` | float | 텐서 네트워크에 사용할 컷오프 값입니다. | 1e-16
`compute_shadows_bias_from_observable` | bool | 클래식 섀도우 측정 프로토콜의 바이어스를 PUB 관측값에 맞출지 여부를 나타내는 불리언 플래그입니다. False인 경우 클래식 섀도우 프로토콜(Z, X, Y를 동일한 확률로 측정)이 사용됩니다. | False
`shadows_bias` | np.ndarray | 무작위 클래식 섀도우 측정 프로토콜에 사용할 바이어스로, 크기 3의 1d 배열 또는 형태 (num_qubits, 3)의 2d 배열입니다. 순서는 ZXY입니다. | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int 또는 `None` | QPU에서의 최대 실행 시간(초)입니다. 런타임이 이 값을 초과하면 작업이 취소됩니다. `None`인 경우 Qiskit Runtime에서 설정한 기본 제한이 적용됩니다. | `None`
`num_randomizations` | int | 노이즈 학습 및 Gate 트월링에 사용할 무작위화 횟수입니다. | 32
`max_layers_to_learn` | int | 학습할 최대 고유 레이어 수입니다. | 4
`mitigate_readout_error` | bool | 판독 오류 완화를 수행할지 여부를 나타내는 불리언 플래그입니다. | True
`num_readout_calibration_shots` | int | 판독 오류 완화에 사용할 샷 수입니다. | 10000
`default_precision` | float | precision이 지정되지 않은 PUB에 사용할 기본 정밀도입니다. | 0.02
`seed` | int 또는 `None` | 재현성을 위해 난수 생성기의 시드를 설정합니다. `None`인 경우 시드를 설정하지 않습니다. | None
## 출력
TEM으로 완화된 결과를 포함하는 Qiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)입니다. 각 PUB의 결과는 다음 필드를 포함하는 [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult)로 반환됩니다:

이름 | 타입 | 설명
-- | -- | --
data | DataBin | TEM으로 완화된 관측값과 표준 오류를 포함하는 Qiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin)입니다. DataBin에는 다음 필드가 있습니다: <ul><li>`evs`: TEM으로 완화된 관측값.</li><li>`stds`: TEM으로 완화된 관측값의 표준 오류.</li></ul>
metadata | dict | 추가 결과를 포함하는 딕셔너리입니다. 딕셔너리에는 다음 키가 포함됩니다: <ul><li>`"evs_non_mitigated"`: 오류 완화 없는 관측값.</li><li>`"stds_non_mitigated"`: 오류 완화 없는 결과의 표준 오류.</li><li>`"evs_mitigated_no_readout_mitigation"`: 오류 완화는 있지만 판독 오류 완화는 없는 관측값.</li><li>`"stds_mitigated_no_readout_mitigation"`: 오류 완화는 있지만 판독 오류 완화는 없는 결과의 표준 오류.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: 오류 완화는 없지만 판독 오류 완화는 있는 관측값.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: 오류 완화는 없지만 판독 오류 완화는 있는 결과의 표준 오류.</li></ul>
## 오류 메시지 가져오기
워크로드 상태가 ERROR인 경우, job.result()를 사용하여 다음과 같이 오류 메시지를 가져오세요: